In [1]:
import pandas as pd
from syslira_tools import PaperLibrary, ScopusClient, ZoteroClient, OpenAlexClient
from dotenv import load_dotenv
import os
from defuzzer import find_all_fuzzy_duplicated_titles
from deduplicator import find_exact_title_duplicates_to_drop

/home/rreider/PycharmProjects/lr-filtering/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# load open alex export


oa_df = pd.read_csv('to_clean.csv', encoding="ISO-8859-1")
oa_df.columns = oa_df.columns.str.replace('ï»¿', '')
print(f"Loaded OpenAlex export with {len(oa_df)} papers")

library_df = oa_df

Loaded OpenAlex export with 416 papers


In [3]:
# remove duplicates


# get and drop title duplicates
title_duplicates = find_exact_title_duplicates_to_drop(library_df)
library_df_deduplicated = library_df.drop(title_duplicates.index)
fuzzy_title_duplicates = find_all_fuzzy_duplicated_titles(library_df_deduplicated, threshold=80, selection_method='human')
library_df_deduplicated = library_df_deduplicated.drop(fuzzy_title_duplicates.index)

# get and drop DOI duplicates (ignore empty DOIs)
doi_duplicates = library_df_deduplicated[library_df_deduplicated.duplicated(subset=["doi"], keep=False) & library_df_deduplicated["doi"].notna() & (library_df_deduplicated["doi"] != "")]
library_df_deduplicated = library_df_deduplicated.drop(doi_duplicates.index)

difference = len(library_df) - len(library_df_deduplicated)
print(f"Dropped {difference} duplicate papers")
library_df = library_df_deduplicated

/home/rreider/PycharmProjects/lr-filtering/cleaning/deduplicator.py:28: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  to_drop_df = pd.concat([to_drop_df, to_drop_from_group])


Duplicate items found:
Item 0: ID: https://openalex.org/W4205807230, Title: Proceedings of the 2021 Conference on Empirical Methods in Natural Language Processing, Type: paratext, Publication Date: 2021-01-01
Item 1: ID: https://openalex.org/W2489487449, Title: Proceedings of the 2020 Conference on Empirical Methods in Natural Language Processing (EMNLP), Type: paratext, Publication Date: 2020-01-01
Invalid input: . Skipping.
Dropped 19 duplicate papers


In [4]:
# drop anything before year 2022
library_df_year = library_df[library_df["publication_year"] >= 2022]
difference = len(library_df) - len(library_df_year)
print(f"Dropped {difference} papers before 2022")
library_df = library_df_year

Dropped 103 papers before 2022


In [5]:
# drop rows that are not a certain article type
valid_types = ['preprint', 'article', 'book-chapter', 'report', 'book', 'dissertation']
library_df_type = library_df[library_df["type"].isin(valid_types)]
difference = len(library_df) - len(library_df_type)
print(f"Dropped {difference} papers of invalid type")
library_df = library_df_type

Dropped 18 papers of invalid type


In [6]:
# remove retracted articles
library_df_no_retracted = library_df[library_df["is_retracted"] == False]
difference = len(library_df) - len(library_df_no_retracted)
print(f"Dropped {difference} retracted papers")
library_df = library_df_no_retracted

Dropped 0 retracted papers


In [7]:
# remove articles not in English
library_df_english = library_df[library_df["language"] == "en"]
difference = len(library_df) - len(library_df_english)
print(f"Dropped {difference} non-English papers")
library_df = library_df_english

Dropped 3 non-English papers


In [8]:
print(f"Final library has {len(library_df)} papers. Removed {len(oa_df) - len(library_df)} papers in total.")

Final library has 273 papers. Removed 143 papers in total.


In [9]:
# export to csv
library_df.to_csv("cleaned.csv", index=False)